# Artifact Evaluation Notebook

Load multiple run folders from one artifact group and compare:

- run-level configuration and global metrics
- per-subject mean/std accuracy
- per-subject-per-fold accuracy
- side-by-side pivots across runs

This notebook assumes each run folder contains files like:

- `config.json`
- `run_metadata.json`
- `global_metrics.json`
- `subject_metrics.json`
- `cv_results.json`


In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)


## 1. Configure the artifact root and the runs to compare

Set `ARTIFACTS_ROOT` to the parent folder that contains all experiment groups.

Example layout:

```text
/artifacts/
    lee-2019-ssvep-fine-tuning/
        20260327_1243_7ce7e84c/
            config.json
            global_metrics.json
            subject_metrics.json
            cv_results.json
            run_metadata.json
```


In [ ]:
WORKING_DIR = Path.cwd().parent.parent

ARTIFACTS_ROOT = str(WORKING_DIR / "artifacts")
ARTIFACT_GROUP = "lee-2019-fine-tuning/"
ARTIFACT_PARADIGM = "SSVEP"

RUN_IDS = [
    "20260331_1704_72960250",
    "20260331_1705_9eb54e90",
    "20260331_1705_910901f0",
    "20260331_1705_a77274e2",
    "20260331_1706_ba14ed26",
    "20260331_1707_21cc859d",
    "20260331_1707_dee8da4b",
    "20260331_1708_89110594",
    "20260331_1708_d12880d7",
    "20260331_1708_f8cb456b",
    "20260331_1709_192f8aee",
    "20260331_1710_ca8f60b9",
    "20260331_1711_a95d0523",
    "20260331_1712_2f1051be",
    "20260331_1714_c64110b4",
    "20260331_1716_3b7f7532",
    "20260331_1717_3f1226fc",
    "20260331_1718_c7b8f566",
    "20260331_1719_19b5a2ff",
    "20260331_1720_ac5f98f5",
    "20260331_1721_88936929",
    "20260331_1723_0b5f0dae",
]

assert Path(ARTIFACTS_ROOT).exists(), f"Artifacts root does not exist: {ARTIFACTS_ROOT}"
assert RUN_IDS, "Please add at least one run id to RUN_IDS"

GROUP_DIR = Path(ARTIFACTS_ROOT) / ARTIFACT_GROUP / ARTIFACT_PARADIGM
assert GROUP_DIR.exists(), f"Artifact group does not exist: {GROUP_DIR}"

print("Artifacts root:", ARTIFACTS_ROOT)
print("Artifact group:", GROUP_DIR)
print("Runs:", RUN_IDS)


## 2. Helpers

In [ ]:
def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def flatten_dict(d, parent_key="", sep="."):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

def load_run_artifacts(group_dir: Path, run_id: str):
    run_dir = group_dir / run_id
    if not run_dir.exists():
        raise FileNotFoundError(f"Run folder not found: {run_dir}")

    files = {
        "config": run_dir / "config.json",
        "run_metadata": run_dir / "run_metadata.json",
        "global_metrics": run_dir / "global_metrics.json",
        "subject_metrics": run_dir / "subject_metrics.json",
        "cv_results": run_dir / "cv_results.json",
    }

    missing = [name for name, path in files.items() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Run {run_id} is missing files: {missing}")

    return {
        "run_id": run_id,
        "run_dir": run_dir,
        "config": read_json(files["config"]),
        "run_metadata": read_json(files["run_metadata"]),
        "global_metrics": read_json(files["global_metrics"]),
        "subject_metrics": read_json(files["subject_metrics"]),
        "cv_results": read_json(files["cv_results"]),
    }

def build_run_summary_row(run):
    row = {
        "run_id": run["run_id"],
        "run_dir": str(run["run_dir"]),
    }

    config = flatten_dict(run["config"])
    metadata = flatten_dict(run["run_metadata"])
    global_metrics = flatten_dict(run["global_metrics"])

    # Prefix columns to avoid collisions
    row.update({f"config.{k}": v for k, v in config.items()})
    row.update({f"meta.{k}": v for k, v in metadata.items()})
    row.update({f"global.{k}": v for k, v in global_metrics.items()})

    return row

def build_subject_rows(run):
    rows = []
    for subject_id, metrics in run["subject_metrics"].items():
        row = {
            "run_id": run["run_id"],
            "subject_id": str(subject_id),
        }
        flat_metrics = flatten_dict(metrics)
        row.update(flat_metrics)
        rows.append(row)
    return rows

def build_fold_rows(run):
    rows = []
    for fold in run["cv_results"]:
        row = {"run_id": run["run_id"]}
        row.update(fold)
        row["subject_id"] = str(row["subject_id"])
        rows.append(row)
    return rows

def sort_run_ids_like_input(df, run_ids):
    if "run_id" in df.columns:
        df = df.copy()
        df["run_id"] = pd.Categorical(df["run_id"], categories=run_ids, ordered=True)
        df = df.sort_values("run_id").reset_index(drop=True)
        df["run_id"] = df["run_id"].astype(str)
    return df


## 3. Load runs

In [ ]:
runs = [load_run_artifacts(GROUP_DIR, run_id) for run_id in RUN_IDS]

print(f"Loaded {len(runs)} runs")
for run in runs:
    print("-", run["run_id"], "->", run["run_dir"])


## 4. Run-level comparison table

This table is useful for comparing configuration differences and overall metrics side by side.


In [ ]:

run_summary_df = pd.DataFrame([build_run_summary_row(run) for run in runs])
run_summary_df = sort_run_ids_like_input(run_summary_df, RUN_IDS)

preferred_cols = [
    "run_id",
    "config.task_name",
    "config.dataset_name",
    "config.seed",
    "config.set_seed",
    # "config.batch_size",
    # "config.learning_rate",
    # "config.n_epochs",
    # "config.early_stopping_patience",
    # "config.sfreq",
    # "config.bandpass_low",
    # "config.bandpass_high",
    # "config.trial_duration_s",
    "global.mean_accuracy",
    # "global.std_accuracy",
    # "global.mean_balanced_accuracy",
    # "global.mean_macro_f1",
]
ordered_cols = [c for c in preferred_cols if c in run_summary_df.columns] + [c for c in run_summary_df.columns if c not in preferred_cols]

display(run_summary_df[ordered_cols])


## 5. Subject-level summary table

One row per `(run, subject)` with mean/std metrics and the raw fold accuracy lists.


In [ ]:

subject_df = pd.DataFrame([row for run in runs for row in build_subject_rows(run)])
subject_df = sort_run_ids_like_input(subject_df, RUN_IDS)
subject_df = subject_df.sort_values(["run_id", "subject_id"]).reset_index(drop=True)

display(subject_df)


## 6. Per-subject accuracy comparison pivot

This is the main side-by-side comparison table for subject means across runs.


In [ ]:

subject_accuracy_pivot = subject_df.pivot(index="subject_id", columns="run_id", values="mean_accuracy")
subject_accuracy_pivot = subject_accuracy_pivot.reindex(sorted(subject_accuracy_pivot.index, key=lambda x: int(x)))
display(subject_accuracy_pivot)


## 7. Per-subject standard deviation pivot

In [ ]:

if "std_accuracy" in subject_df.columns:
    subject_std_pivot = subject_df.pivot(index="subject_id", columns="run_id", values="std_accuracy")
    subject_std_pivot = subject_std_pivot.reindex(sorted(subject_std_pivot.index, key=lambda x: int(x)))
    display(subject_std_pivot)


## 8. Fold-level comparison table

One row per `(run, subject, fold)`.


In [ ]:

fold_df = pd.DataFrame([row for run in runs for row in build_fold_rows(run)])
fold_df = sort_run_ids_like_input(fold_df, RUN_IDS)
fold_df = fold_df.sort_values(["run_id", "subject_id", "fold_id"]).reset_index(drop=True)

display_cols = [
    "run_id",
    "subject_id",
    "fold_id",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "best_valid_loss",
    "n_train",
    "n_val",
    "n_test",
]
display(fold_df[[c for c in display_cols if c in fold_df.columns]])


## 9. Per-subject-per-fold accuracy pivot

Rows are `(subject_id, fold_id)` and columns are runs.


In [ ]:

fold_accuracy_pivot = fold_df.pivot_table(
    index=["subject_id", "fold_id"],
    columns="run_id",
    values="accuracy",
    aggfunc="first",
).sort_index()

display(fold_accuracy_pivot)


## 10. Delta tables relative to a baseline run

Pick one run as the baseline and compare all others against it.


In [ ]:

BASELINE_RUN_ID = RUN_IDS[0]

subject_delta_vs_baseline = subject_accuracy_pivot.subtract(subject_accuracy_pivot[BASELINE_RUN_ID], axis=0)
display(subject_delta_vs_baseline)

fold_delta_vs_baseline = fold_accuracy_pivot.subtract(fold_accuracy_pivot[BASELINE_RUN_ID], axis=0)
display(fold_delta_vs_baseline)


## 11. Optional: compact long-form export tables

Useful if you want to save CSV files for thesis/report work.


In [ ]:

EXPORT_DIR = GROUP_DIR / "_comparison_exports"
EXPORT_DIR.mkdir(exist_ok=True)

run_summary_df.to_csv(EXPORT_DIR / "run_summary.csv", index=False)
subject_df.to_csv(EXPORT_DIR / "subject_summary_long.csv", index=False)
fold_df.to_csv(EXPORT_DIR / "fold_summary_long.csv", index=False)
subject_accuracy_pivot.to_csv(EXPORT_DIR / "subject_accuracy_pivot.csv")
fold_accuracy_pivot.to_csv(EXPORT_DIR / "fold_accuracy_pivot.csv")

print("Exported comparison tables to:", EXPORT_DIR)
